In [1]:
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# Sample dictionary
data = {
    "Category": ["A", "B", "C", "D"],
    "Value": [10, 15, 7, 20]
}


In [3]:

# Create horizontal bar chart
fig = px.bar(
    data,
    x="Value",       # x-axis will be the values
    y="Category",    # y-axis will be the categories
    orientation="h", # horizontal bars
    title="Horizontal Bar Chart Example"
)


In [4]:

# Thickness controls (optional)
bar_px = 50
top_bottom_margin = 120
n = len(data["Category"])
fig.update_layout(
    height=top_bottom_margin + bar_px * n,
    bargap=0.8
)


In [5]:

# Add open-circle markers at the bar tips
fig.add_trace(
    go.Scatter(
        x=data["Value"],
        y=data["Category"],
        mode="markers",
        marker=dict(
            symbol="circle",   # open circle; use 'circle' for filled
            size=10,
            # line=dict(width=2, color="black"),
            color="rgba(0,0,0,1)"  # transparent fill for safety
        ),
        showlegend=False,
        hoverinfo="skip"
    )
)


In [6]:
# %%
import pandas as pd
import plotly.express as px

# %%
# Create a DataFrame (instead of a dict)
df = pd.DataFrame({
    "Category": ["A", "B", "C", "D"],
    "Value": [10, 15, 7, 20]
})

# %%
# Horizontal bar chart with 'o' as outside text
fig = px.bar(
    df,
    x="Value",
    y="Category",
    orientation="h",
    title="Horizontal Bar Chart with 'o' at Tip",
    text=["o"] * len(df)  # one 'o' per row
)

# Thickness controls (optional)
bar_px = 50
top_bottom_margin = 120
fig.update_layout(
    height=top_bottom_margin + bar_px * len(df),
    bargap=0.15,
    margin=dict(r=60)  # avoid clipping the outside text
)

# Place the 'o' just beyond the bar tip
fig.update_traces(
    textposition="outside",
    textfont=dict(size=18, color="black")
)

# Ensure space on the right for the outside text
max_val = df["Value"].max()
fig.update_xaxes(range=[0, max_val * 1.10])

fig.show()

In [7]:
import duckdb
from collections import defaultdict

con = duckdb.connect('change_tracker.db')


In [8]:
drivers_question_data = con.sql(
                        """
                        SELECT
                            -- q.id AS question_id,
                            d.drivers_name,
                            q.questions,
                            CAST(a.answers AS INTEGER) AS answers
                        FROM answers a
                            JOIN users u      ON u.id = a.user_id
                            JOIN questions q  ON q.id = a.questions_id
                            JOIN drivers d    ON d.id = q.drivers_id
                        WHERE u.username = 'john_doe'
                            QUALIFY ROW_NUMBER() OVER (
                            PARTITION BY q.id
                            ORDER BY a.modified_time DESC, a.id DESC
                        ) = 1
                        """
                        ).fetchall()

In [9]:

denormalized_drivers_question_data = defaultdict(list)

for driver, question, answer in drivers_question_data:
    denormalized_drivers_question_data[driver].append({
        "question": question,
        "answer": answer
    })

denormalized_drivers_question_data = dict(denormalized_drivers_question_data)

In [10]:
def figure(title: str, data: dict):
    # Convert to DataFrame
    df = pd.DataFrame({
        "Metric": list(dict(data).keys()),
        "Score": list(dict(data).values())
    })

    fig = px.bar(
        df,
        x="Score",       # x-axis will be the values
        y="Metric",    # y-axis will be the categories
        orientation="h", # horizontal bars
        title=title
    )
    bar_px = 50
    top_bottom_margin = 120
    n = len(df["Metric"])
    fig.update_layout(
        height=top_bottom_margin + bar_px * n,
        bargap=0.8
    )
    fig.add_trace(
        go.Scatter(
            x=df["Score"],
            y=df["Metric"],
            mode="markers",
            marker=dict(
                symbol="circle",   # open circle; use 'circle' for filled
                size=10,
                # line=dict(width=2, color="black"),
                color="rgba(0,0,0,1)"  # transparent fill for safety
            ),
            showlegend=False,
            hoverinfo="skip"
        )
    )
    fig.update_xaxes(range=[0, 5])

    return fig

In [ ]:
(denormalized_drivers_question_data['Business Performance'])

[{'question': "Your Department's (i.e. Operations, Merchandising etc.) effectiveness",
  'answer': 1},
 {'question': 'Managing costs and resources in your Team', 'answer': 4},
 {'question': 'The level of customer service (internal or external) your Team provides',
  'answer': 3}]

: 

In [17]:
df = pd.DataFrame({
        "Metric": list(denormalized_drivers_question_data['Business Performance'].keys()),
        "Score": list(denormalized_drivers_question_data['Business Performance'].values())
    })
df

AttributeError: 'list' object has no attribute 'keys'

In [11]:
figure_created = figure("nothing", denormalized_drivers_question_data['Business Performance'])

import plotly.io as pio

# Show the figure using Plotly's IO
pio.show(figure_created)